<h2>🧪 Quick Sanity Check</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Optional mini-cell to confirm the kernel is alive ✅
</div>

<h1>⚙️ Setup</h1>

<div style="padding: 10px 12px; border-left: 6px solid #F59E0B; background: #d6b941ff; border-radius: 10px;">
Install + upgrade key libraries (UnsLoTh + TRL + Transformers).<br>
If you’re on Colab/Kaggle, this cell handles the heavy lifting 🧰
</div>

In [1]:
# %%capture
# import os, importlib.util
# !pip install --upgrade -qqq uv
# if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
#     try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
#     except: get_numpy = "numpy"; get_pil = "pillow"
#     !uv pip install -qqq \
#         "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
#         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#         "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#         git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# elif importlib.util.find_spec("unsloth") is None:
#     !uv pip install -qqq unsloth
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [2]:
# !pip install -U bitsandbytes

In [3]:
try:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

except:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-py-3-12/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

Using Python 3.12.12 environment at: /usr
  × No solution found when resolving dependencies:
  ╰─▶ Because torch==2.9.1 has no wheels with a matching Python ABI tag (e.g.,
      `cp312`) and xformers==0.0.33.post2 depends on torch==2.9.1, we can
      conclude that xformers==0.0.33.post2 cannot be used.
      And because only xformers{(platform_machine == 'AMD64' and 'linux'
      in sys_platform) or (platform_machine == 'x86_64' and 'linux' in
      sys_platform) or (platform_machine == 'AMD64' and sys_platform
      == 'win32') or (platform_machine == 'x86_64' and sys_platform
      == 'win32')}==0.0.33.post2 is available and unsloth==2026.1.2
      depends on xformers{(platform_machine == 'AMD64' and 'linux' in
      sys_platform) or (platform_machine == 'x86_64' and 'linux' in
      sys_platform) or (platform_machine == 'AMD64' and sys_platform
      == 'win32') or (platform_machine == 'x86_64' and sys_platform ==
      'win32')}==0.0.33.post2, we can conclude that unsloth==2026.1.

2026-02-13 10:21:43.147345: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770978103.468550      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770978103.571153      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770978104.343670      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770978104.343702      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770978104.343703      44 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


<h1 style="color: #3B82F6;">🤖 Model</h1>

<h2 style="color: #3B82F6;">📥 Load Base Model</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Loads <b>unsloth/gpt-oss-20b</b> in 4-bit for VRAM-friendly training 🧊
</div>

In [4]:
print ("done")

done


In [5]:
# import kagglehub 
# path = kagglehub.model_download("barnobarno/gpt-oss-20b/transformers/unsloth")

# print("Path to model files:", path)

In [6]:
# %%capture
# import os, importlib.util
# !pip install --upgrade -qqq uv
# if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
#     try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
#     except: get_numpy = "numpy"; get_pil = "pillow"
#     !uv pip install -qqq \
#         "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
#         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#         "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#         git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# elif importlib.util.find_spec("unsloth") is None:
#     !uv pip install -qqq unsloth
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [7]:
import torch
max_seq_length = 4096   # Reduced from 2048 - speeds up generation significantly
 # Larger rank = smarter, but slower
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/input/gpt-oss-20b-bnb-4bit/transformers/unsloth/1" ,
    max_seq_length = max_seq_length,
    local_files_only =  True ,
    load_in_4bit=True , 
    dtype=None
)

==((====))==  Unsloth 2026.1.3: Fast Gpt_Oss patching. Transformers: 4.57.1.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
lora_rank = 16

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"


In [10]:
import re


<h2 style="color: #265ccfff;">🧩 LoRA Settings</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #654ed5ff; border-radius: 10px;">
Pick a LoRA rank. Higher = more capacity, slower + more memory 🔧
</div>

<h1 style="color: #1757ebff;">📚 Data</h1>

<h2>🧾 Load OlymMATH (en-hard)</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #ECFEFF; border-radius: 10px;">
Pulls the dataset + previews it in a DataFrame 🗂️
</div>

In [11]:
# from datasets import load_dataset

# # Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset("RUC-AIBOX/OlymMATH", "en-hard")

In [12]:
import pandas as pd 
data = pd.read_csv("/kaggle/input/olymmath-hard/OlymMATH-EN-EASY.csv") # pd.DataFrame(ds['test'])
print(data.head())


                                             problem         answer  \
0  Given a non-negative integer sequence $\{a_n\}...            948   
1  Given that $AB$ is a diameter of circle $\odot...  4\sqrt{15}-14   
2  Calculate the value of $\sqrt{9+8\cos 20^{\cir...              3   
3  A sphere is circumscribed around tetrahedron $...   \frac{18}{5}   
4  Find the minimum value of $f(x) = \sum_{i=1}^{...      801730806   

         subject           unique_id  
0  Combinatorics  OlymMATH-EASY-0-EN  
1       Geometry  OlymMATH-EASY-1-EN  
2        Algebra  OlymMATH-EASY-2-EN  
3       Geometry  OlymMATH-EASY-3-EN  
4        Algebra  OlymMATH-EASY-4-EN  


<h1 style="color: #b0de4eff;">🧠 Prompting</h1>

<h2 style="color: #3e1492ff;">📝 System + User Prompt Template</h2>

<div style="padding: 10px 12px; border-left: 6px solid #22C55E; background: #F0FDF4; border-radius: 10px;">
Defines the instruction style: step-by-step reasoning + final integer inside <b>\\boxed{}</b> ✅
</div>

In [13]:
SYSTEM_PROMPT = """You are a math problem solver. You MUST solve problems by writing Python code.
Write your solution in a ```python``` code block. The code should compute and print the final answer using print('\\boxed{answer}').***IMPORTANT***: DO NOT CALL ANY PYTHON TOOL. YOU DO NOT HAVE ACESS TO ANY TOOL. ONLY WRITE YOUR CODE IN THE CORRECT FORMAT AS STATED .  """

def format_prompt(question):
    return [
        {"role": "user", "content": f"{question}\n\nSolve this step by step Using a python code. Write your solution in a ```python``` code block that print the final answer as \\boxed{{answer}}. "}
    ]

<h2 style="color: #e323a9ff;">🔌 LoRA: Add Adapters</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Attaches LoRA modules to attention + MLP layers (fast, memory-friendly fine-tuning) 🧬
</div>

In [14]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA adapters added successfully!")

Unsloth: Making `model.base_model.model.model` require gradients
LoRA adapters added successfully!


<h2 style="color: #e323a9ff;">🧪 Build a Tiny Training Set</h2>

<div style="padding: 10px 12px; border-left: 6px solid #06B6D4; background: #ECFEFF; border-radius: 10px;">
Creates a small list of prompt/answer pairs for quick testing 🧫
</div>

In [15]:
from sklearn.model_selection import train_test_split

# Sample 10 datapoints, stratified by subject
# train_size=10 guarantees exactly 10 samples
# stratify=data['subject'] ensures the subject distribution matches the full dataset
sampled_data, _ = train_test_split(
    data, 
    train_size=10, 
    stratify=data['subject'], 
    random_state=42  # Fixed seed for reproducibility
)

# Create training dataset from the sampled data
train_data = []
for _, row in sampled_data.iterrows():
    question = row['problem']
    answer = row['answer']  # The ground truth answer
    
    train_data.append({
        "prompt": format_prompt(question),
        "answer": str(answer),
        "reasoning_effort": "low"
    })

print(f"Created dataset with {len(train_data)} problems")
if len(train_data) > 0:
    print(f"Sample question: {train_data[0]['prompt'][0]['content'][:200]}...")
    print(f"Sample answer: {train_data[0]['answer']}")

Created dataset with 10 problems
Sample question: Let $x_{i} \geq 0 (i = 1, 2, \cdots, 6)$, and satisfy $\begin{cases} x_{1} + x_{2} + \cdots + x_{6} = 1, \\ x_{1} x_{3} x_{5} + x_{2} x_{4} x_{6} \geq \frac{1}{540} \end{cases}$. Find the maximum valu...
Sample answer: \frac{19}{540}


<h1 style="color: #e323a9ff;">🧰 Parsing Utilities</h1>

<h2 style="color: #ffffffff;">📦 NO NEED TO CHANGE <span style="color:#4F46E5;"><b></b></span></h2>

<div style="padding: 10px 12px; border-left: 6px solid #4F46E5; background: #EEF2FF; border-radius: 10px;">
We score outputs by pulling the final value from <b>\\boxed{...}</b> 🔍
</div>

In [16]:
"""LOOKS OK HERE"""




# Answer extraction function
def extract_boxed_answer(text):
    """Extract the answer from \\boxed{} in the model output"""
    # Try to find \boxed{...}
    patterns = [
        r'\\boxed\{([^{}]*)\}',  # Simple \boxed{answer}
        r'\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}',  # Nested braces
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            # Return the last match (final answer)
            answer_str = matches[-1].strip()
            try:
                # Try to parse as number
                # Handle fractions, negatives, etc.
                answer_str = answer_str.replace(',', '')  # Remove commas
                if '/' in answer_str:
                    # Handle fractions
                    parts = answer_str.split('/')
                    return float(parts[0]) / float(parts[1])
                return float(answer_str)
            except:
                return None
    return None
"""The above function looks ok , keep as is"""



def parse_true_answer(answer_str):
    """Parse the ground truth answer to a number"""
    try:
        answer_str = str(answer_str).strip().replace(',', '')
        if '/' in answer_str:
            parts = answer_str.split('/')
            return float(parts[0]) / float(parts[1])
        return float(answer_str)
    except:
        return None

# Test extraction
test_text = "The answer is \\boxed{42}"
print(f"Test extraction: {extract_boxed_answer(test_text)}")

Test extraction: 42.0


<h1 style="color: #e323a9ff;">🎯 Rewards</h1>

<h2 style="color: #ffffffff;">✨ I SHOULD ADD A HARMONY REWARD + A TOKEN USAGE REWARD</h2>

<div style="padding: 10px 12px; border-left: 6px solid #F97316; background: #d392b3ff; border-radius: 10px;">
Two rewards:
<ul style="margin: 6px 0 0 18px;">
  <li><b>Format</b> reward for including <b>\\boxed{}</b> 🧾</li>
  <li><b>Answer</b> reward based on distance from the true value 🎯</li>
</ul>
</div>

In [17]:
"""THIS FUNCTION NEEDS TO BE TURNED INTO REWARD FUNCTION"""

# def repair_harmony_tokens(self, token_ids: list[int]) -> list[int]:
#         """
#         Robustly repairs malformed Harmony messages where <|message|> is skipped.
#         Specifically handles 'commentary' split into two tokens.
#         """
#         CHANNEL_TOKEN = 200005
#         MESSAGE_TOKEN = 200008
        
#         new_ids = []
#         i = 0
#         n = len(token_ids)
        
#         while i < n:
#             token = token_ids[i]
#             new_ids.append(token)
            
#             # Trigger only on <|channel|>
#             if token == CHANNEL_TOKEN:
#                 # Default channel length is 1 token
#                 channel_len = 1
                
#                 # Check for special case: 'commentary' split into two tokens
#                 if i + 2 < n:
#                     try:
#                         # Decode next two tokens
#                         chunk = token_ids[i+1 : i+3]
#                         decoded_chunk = self.tokenizer.decode(chunk).lower()
#                         if "commentary" in decoded_chunk:
#                             channel_len = 2
#                     except Exception:
#                         pass
                
#                 # Copy the channel tokens
#                 for _ in range(channel_len):
#                     if i + 1 < n:
#                         i += 1
#                         new_ids.append(token_ids[i])
                
#                 # Check if the NEXT token is <|message|>. If not, inject it.
#                 if i + 1 < n:
#                      if token_ids[i+1] != MESSAGE_TOKEN:
#                         new_ids.append(MESSAGE_TOKEN)
#                 else: 
#                      # End of sequence, better close with message token so parser doesn't fail
#                      new_ids.append(MESSAGE_TOKEN)
            
#             i += 1
            
#         return new_ids

'THIS FUNCTION NEEDS TO BE TURNED INTO REWARD FUNCTION'

In [18]:
# Reward Functions for RLVR
import re

def format_reward(completions, **kwargs):
    """Reward for proper formatting with \\boxed{}"""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        # Check if response contains \\boxed{}
        if "\\boxed{" in response:
            scores.append(0.5)  # Small bonus for correct format
        else:
            scores.append(-0.5)  # Penalty for missing format
    return scores








def answer_reward(completions, answer, **kwargs):
    """
    Main reward: negative distance from correct answer
    Reward = -1 * abs(TRUE - PREDICTED) / abs(TRUE) if TRUE != 0
    Reward = -1 * abs(PREDICTED) if TRUE == 0
    Reward = 0 if exactly correct
    """
    scores = []
    
    for completion, true_ans in zip(completions, answer):
        response = completion[0]["content"]
        predicted = extract_boxed_answer(response)
        true_value = parse_true_answer(true_ans)
        
        if predicted is None or true_value is None:
            # Can't parse answer - give penalty
            scores.append(-2.0)
            continue
        
        # Calculate distance-based reward
        if abs(predicted - true_value) < 1e-6:
            # Exactly correct!
            reward = 10.0  # Bonus for correct answer
        else:
            # Calculate relative error
            if abs(true_value) > 1e-6:
                relative_error = abs(true_value - predicted) / abs(true_value)
            else:
                relative_error = abs(predicted)
            
            # Negative reward based on error, capped at -2
            reward = -1.0 * min(relative_error, 2.0)
        
        scores.append(reward)
    
    return scores

# Test the reward function
test_completions = [[{"content": "The answer is \\boxed{42}"}]]
test_answers = ["42"]
print(f"Test reward (correct): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{40}"}]]
print(f"Test reward (close): {answer_reward(test_completions, test_answers)}")

test_completions = [[{"content": "The answer is \\boxed{0}"}]]
print(f"Test reward (far): {answer_reward(test_completions, test_answers)}")

Test reward (correct): [10.0]
Test reward (close): [-0.047619047619047616]
Test reward (far): [-1.0]


In [19]:
import re
import sys
import io
import math
import contextlib
import threading
import numpy as np

# --- Helper 1: Extract the Python Code ---
def extract_code_block(text):
    """
    Finds the last Python block in the text.
    Returns None if no code block is found.
    """
    pattern = r"```python\n(.*?)\n```"
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1].strip()
    return None

# --- Helper 2: Safe Execution Sandbox with Timeout ---
def unsafe_execute_and_capture(code, timeout=60):
    """
    Executes code in a fresh, isolated namespace and captures stdout.
    Includes a 60-second timeout to prevent infinite loops.
    """
    result = {"output": "", "success": False}
    
    def run_code():
        # 1. Fresh Namespace (Isolation)
        # We only allow standard math/logic libraries. 
        # NO access to 'answer' or 'gt' variables.
        safe_globals = {
            "math": math,
            "np": np,
            "numpy": np,
            "print": print,
            "range": range,
            "len": len,
            "int": int,
            "float": float,
            "str": str,
            "list": list,
            "set": set,
            "dict": dict,
            "tuple": tuple,
            "abs": abs,
            "min": min,
            "max": max,
            "sum": sum,
            "pow": pow,
            "round": round,
            "enumerate": enumerate,
            "zip": zip,
            "sorted": sorted,
            "reversed": reversed,
            "map": map,
            "filter": filter,
            "all": all,
            "any": any,
            "True": True,
            "False": False,
            "None": None,
            "isinstance": isinstance,
            "type": type,
            "bool": bool,
            "divmod": divmod,
            "__builtins__": {
                "True": True,
                "False": False,
                "None": None,
                "isinstance": isinstance,
                "type": type,
                "len": len,
                "range": range,
                "print": print,
                "int": int,
                "float": float,
                "str": str,
                "bool": bool,
                "list": list,
                "dict": dict,
                "set": set,
                "tuple": tuple,
                "abs": abs,
                "min": min,
                "max": max,
                "sum": sum,
                "pow": pow,
                "round": round,
                "sorted": sorted,
                "reversed": reversed,
                "enumerate": enumerate,
                "zip": zip,
                "map": map,
                "filter": filter,
                "all": all,
                "any": any,
                "divmod": divmod,
            },
        }
        
        # 2. Capture Stdout
        output_buffer = io.StringIO()
        
        try:
            # Redirect stdout to our buffer
            with contextlib.redirect_stdout(output_buffer):
                exec(code, safe_globals)
            result["output"] = output_buffer.getvalue().strip()
            result["success"] = True
        except Exception as e:
            result["output"] = f"Error: {str(e)}"
            result["success"] = False
    
    # Run with timeout using threading
    thread = threading.Thread(target=run_code)
    thread.daemon = True
    thread.start()
    thread.join(timeout=timeout)
    
    if thread.is_alive():
        # Timeout occurred
        return "Error: Execution timed out (60s limit)", False
    
    return result["output"], result["success"]

# --- Helper 3: Answer Parsing ---
def parse_output(text):
    """
    Extracts the value inside \\boxed{...} from the execution output.
    """
    # Standard boxed pattern
    matches = re.findall(r"\\\\boxed\{([^{}]*)\}", text)
    if matches:
        return matches[-1]
    
    # Fallback: often models just print the number at the end
    # If the output is just a number, take it.
    try:
        return float(text.strip())
    except:
        pass
        
    return None

# --- Main Reward Function ---
def robust_code_reward_func(completions, answer, **kwargs):
    """
    The STRICT Reward Function.
    - No Code = -2.0 (Hard Fail)
    - Crash = -1.0 (Fail)
    - Wrong Answer = -0.5 (Soft Fail)
    - Correct Answer = +2.0 (Success)
    """
    rewards = []
    
    for completion, gt_str in zip(completions, answer):
        text = completion[0]["content"]
        
        # 1. Check for Code Block
        code = extract_code_block(text)
        if not code:
            rewards.append(-2.0)  # HEAVY PENALTY: Did not attempt to code
            continue
            
        # 2. Execute Code
        exec_output, success = unsafe_execute_and_capture(code)
        
        if not success:
            rewards.append(-1.0)  # PENALTY: Code crashed (syntax error, etc)
            continue
            
        # 3. Parse and Compare Answer
        try:
            # Parse Ground Truth
            gt_val = float(str(gt_str).replace(',', ''))
            
            # Parse Model Output
            pred_str = parse_output(exec_output)
            if pred_str is None:
                rewards.append(-0.5)  # Penalty: Ran code but printed no answer
                continue
                
            pred_val = float(str(pred_str).replace(',', ''))
            
            # 4. Final Verification (Tolerance 1e-4)
            if abs(pred_val - gt_val) < 1e-4:
                rewards.append(2.0)  # JACKPOT: Correct Answer
            else:
                rewards.append(-0.5)  # Penalty: Wrong Answer
                
        except Exception:
            # If parsing fails (e.g. model printed "x=5" instead of "5")
            rewards.append(-0.5)
            
    return rewards

In [20]:
# --- Test / Verification Block ---

# 1. Mock Data: simulate what the model might output
test_cases = [
    # Case A: Perfect Code (Correct Answer)
    {
        "content": "To solve this, I will use Python.\n```python\nprint('\\\\boxed{42}')\n```",
        "ground_truth": "42",
        "expected": 2.0,
        "desc": "Correct Code"
    },
    # Case B: Code Runs, but Answer is Wrong
    {
        "content": "Calculated it is 10.\n```python\nprint('\\\\boxed{10}')\n```",
        "ground_truth": "42",
        "expected": -0.5,
        "desc": "Wrong Answer"
    },
    # Case C: Syntax Error / Crash
    {
        "content": "Let me divide by zero.\n```python\nprint(1/0)\n```",
        "ground_truth": "42",
        "expected": -1.0,
        "desc": "Code Crash"
    },
    # Case D: Code Runs, but prints no answer
    {
        "content": "I calculated it but forgot to print.\n```python\nx = 42\n```",
        "ground_truth": "42",
        "expected": -0.5,
        "desc": "No Output"
    },
    # Case E: No Code Block at all
    {
        "content": "The answer is 42. I didn't use python.",
        "ground_truth": "42",
        "expected": -2.0,
        "desc": "No Code Block"
    }
]

# 2. Format for the Reward Function
# The function expects list of completions [[{content}]] and list of answers
completions = [[{"content": t["content"]}] for t in test_cases]
answers = [t["ground_truth"] for t in test_cases]

# 3. Run the Reward Function
print(f"{'Description':<20} | {'Exp':<5} | {'Got':<5} | {'Status'}")
print("-" * 50)

results = robust_code_reward_func(completions, answers)

for i, score in enumerate(results):
    expected = test_cases[i]["expected"]
    status = "✅ PASS" if score == expected else "❌ FAIL"
    print(f"{test_cases[i]['desc']:<20} | {expected:<5} | {score:<5} | {status}")

Description          | Exp   | Got   | Status
--------------------------------------------------
Correct Code         | 2.0   | -0.5  | ❌ FAIL
Wrong Answer         | -0.5  | -0.5  | ✅ PASS
Code Crash           | -1.0  | -1.0  | ✅ PASS
No Output            | -0.5  | -0.5  | ✅ PASS
No Code Block        | -2.0  | -2.0  | ✅ PASS


<h1 style="color:  #e323a9ff;">🗃️ Dataset</h1>

<h2 style="color: #e323a9ff;">🔁 Replicate Samples for More Steps</h2>

<div style="padding: 10px 12px; border-left: 6px solid #58d987ff; background: #F0FDF4; border-radius: 10px;">
Creates a Hugging Face <b>Dataset</b> and replicates entries so GRPO can run for many steps 🧩
</div>

In [21]:
# Create the HuggingFace Dataset
from datasets import Dataset

# Replicate the 50 problems to have enough data for 100 steps
# With batch_size=1 and 100 steps, we need at least 100 samples
replicated_data =  train_data * 1  # 50 problems * 2 = 100 samples

dataset = Dataset.from_list(replicated_data)

# Calculate prompt length for configuration
sample_prompt = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="low"
)
max_prompt_length = len(tokenizer(sample_prompt)["input_ids"]) + 10  # Add buffer

print(f"Dataset size: {len(dataset)}")
print(f"Max prompt length: {max_prompt_length}")
print(f"Sample formatted prompt:\n{sample_prompt[:500]}...")

Dataset size: 10
Max prompt length: 306
Sample formatted prompt:
<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-02-13

Reasoning: low

# Valid channels: analysis, commentary, final. Channel must be included for every message.
Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Let $x_{i} \geq 0 (i = 1, 2, \cdots, 6)$, and satisfy $\begin{cases} x_{1} + x_{2} + \cdots + x_{6} = 1, \\ x_{1} x_{3} x_{5} + x_{2} x_{4} x_{6} \geq \frac{...


<h1 style="color:  #e323a9ff;">🧪 GRPO</h1>

<h2 style="color: #e323a9ff;">⚙️ need to understand GRPO</h2>

<div style="padding: 10px 12px; border-left: 6px solid #3B82F6; background: #EFF6FF; border-radius: 10px;">
Sets sampling + optimization hyperparams.<br>
Watch <b>max_prompt_length</b> + <b>max_completion_length</b> to avoid truncation ✂️
</div>

In [22]:
# !uv pip install wandb 

In [23]:
# import wandb
# wandb.login(key="YOUR_WANDB_API_KEY")

# run = wandb.init(
#     entity="barnoahmed666-none",
#     project="OSS GRPO ",
#     config={
#         "model_name": "unsloth/gpt-oss-20b",
#         "lora_rank": lora_rank,
#         "max_seq_length": max_seq_length,
#         "learning_rate": 5e-5,
#         "weight_decay": 0.001,
#         "warmup_ratio": 0.1,
#         "lr_scheduler_type": "linear",
#         "optim": "adamw_8bit",
#         "per_device_train_batch_size": 2,
#         "gradient_accumulation_steps": 1,
#         "num_generations": 2,
#         "max_steps": 100,
#         "temperature": 1.0,
#     },
# )


In [24]:
# Configure GRPO Training
from trl import GRPOConfig, GRPOTrainer
import gc

# Cap completion length to something reasonable for speed
max_completion_length = 2048    #min(max_seq_length - max_prompt_length, 512)  # Cap at 512 tokens

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 5e-5,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 1,
    num_generations = 4 ,  # Number of completions per prompt
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 10,  # 100 steps for testing
    #save_steps = 50,
    report_to = "none",
    output_dir = "outputs_grpo_test",
    #beta=0.01,
)

print(f"Max completion length: {max_completion_length}")
print("Training config ready!")

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 4
Max completion length: 2048
Training config ready!


<h2 style="color: #e323a9ff;">🏗️ Build the Trainer</h2>

<div style="padding: 10px 12px; border-left: 6px solid #8B5CF6; background: #F5F3FF; border-radius: 10px;">
Wires together: <b>model</b> + <b>tokenizer</b> + <b>reward functions</b> + <b>dataset</b> 🧷
</div>

In [25]:
# Initialize the GRPO Trainer
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        robust_code_reward_func
       # format_reward,    # Reward for using \boxed{}
       # answer_reward,    # Main reward: distance-based correctness
    ],
    args = training_args,
    train_dataset = dataset,
)

print("Trainer initialized!")

Trainer initialized!


<h1 style="color: #e323a9ff;">🚀 Training Run</h1>

<h2 style="color: #e323a9ff;">🏋️ GRPO Train</h2>

<div style="padding: 10px 12px; border-left: 6px solid #EF4444; background: #FEF2F2; border-radius: 10px;">
Heads up: generation dominates runtime ⏳<br>
If it’s too slow, reduce <b>max_steps</b>, <b>num_generations</b>, or <b>max_new_tokens</b>.
</div>

In [26]:
# Start training - 100 steps
# Monitor the 'reward' column in the output table - it should increase over time
import time

print("Starting training... (This will take a while - generation is the slow part)")
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"\n{'='*50}")
print(f"Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"Time per step: {training_time/100:.2f} seconds")
print(f"Estimated time for 1000 steps: {(training_time/100)*1000/60:.2f} minutes")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting training... (This will take a while - generation is the slow part)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 7,962,624 of 20,922,719,808 (0.04% trained)
`generation_config` default values have been modified to match model-specific defaults: {'max_length': 131072}. If this is not desired, please set these values explicitly.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / robust_code_reward_func / mean,rewards / robust_code_reward_func / std
1,0.000000,-2.000000,0.000000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0.000000,-2.000000,0.000000
2,-0.000000,-1.625000,0.750000,1795.750000,1039.000000,2048.000000,0.750000,1039.000000,1039.000000,1039.000000,No Log,No Log,No Log,No Log,No Log,0.000000,-1.625000,0.750000
3,0.000000,-2.000000,0.000000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.000565,-2.000000,0.000000
4,0.000000,-1.625000,0.750000,1936.250000,1601.000000,2048.000000,0.750000,1601.000000,1601.000000,1601.000000,No Log,No Log,No Log,No Log,No Log,0.001303,-1.625000,0.750000
5,0.000000,-2.000000,0.000000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.000889,-2.000000,0.000000
6,0.000000,-1.250000,0.500000,1870.750000,1750.000000,2048.000000,0.250000,1811.666748,1750.000000,1875.000000,No Log,No Log,No Log,No Log,No Log,0.001213,-1.250000,0.500000
7,0.000000,-2.000000,0.000000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.001453,-2.000000,0.000000
8,0.000000,-1.750000,0.500000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.001302,-1.750000,0.500000
9,0.000000,-1.750000,0.500000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.001882,-1.750000,0.500000
10,0.000000,-2.000000,0.000000,2048.000000,2048.000000,2048.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.001154,-2.000000,0.000000


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).



Training completed in 1770.96 seconds (29.52 minutes)
Time per step: 17.71 seconds
Estimated time for 1000 steps: 295.16 minutes


<h1 style="color: #e323a9ff;">🔎 Evaluation</h1>

<h2 style="color: #e323a9ff;">🧠 Quick Inference Smoke Test</h2>

<div style="padding: 10px 12px; border-left: 6px solid #10B981; background: #ECFDF5; border-radius: 10px;">
Generates on a known training prompt and prints the model output 🧪
</div>

In [27]:
# Test inference after training
text = tokenizer.apply_chat_template(
    train_data[0]["prompt"],
    tokenize = False,
    add_generation_prompt = True,
    reasoning_effort = "low",
)

from transformers import TextStreamer

print("Testing trained model on first problem:")
print("="*50)
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)
print(f"\nExpected answer: {train_data[0]['answer']}")

Testing trained model on first problem:
<|channel|>analysis<|message|>We need maximize sum of cyclic triple products with constraint sum=1 and product of odd indices times even indices >=1/540. Use brute force? Use random search or optimization. Let's approximate. We'll write python code to search grid.<|end|><|start|>assistant<|channel|>commentary to=functions code<|message|>{"cmd":["bash","-lc","python3 - << 'PY'\nimport random, math\nbest=0\nbestx=None\nfor _ in range(2000000):\n    xs=[random.random() for _ in range(6)]\n    s=sum(xs)\n    xs=[x/s for x in xs]\n    if xs[0]*xs[2]*xs[4]+xs[1]*xs[3]*xs[5] >= 1/540:\n        val=sum(xs[i]*xs[(i+1)%6]*xs[(i+2)%6] for i in range(6))\n        if val>best:\n            best=val; bestx=xs\nprint(best,bestx)\nPY"]}<|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call|><|call

In [28]:
# Save the LoRA adapters
model.save_pretrained("gpt_oss_120b_math_lora_test")
tokenizer.save_pretrained("gpt_oss_120b_math_lora_test")
print("LoRA adapters saved to 'gpt_oss_20b_math_lora_test'")

LoRA adapters saved to 'gpt_oss_20b_math_lora_test'


## Optional: Save merged model or push to Hub

Uncomment the options below as needed:
- **LoRA only**: Smallest size, requires base model to load
- **Merged 16bit**: Full model in fp16
- **MXFP4**: GPT-OSS native precision, good for VLLM

In [29]:
# Optional: Merge and save in different formats

# Save merged model in MXFP4 (GPT-OSS native precision)
# model.save_pretrained_merged("gpt_oss_20b_math_mxfp4", tokenizer, save_method="mxfp4")

# Save merged model in 16bit
# model.save_pretrained_merged("gpt_oss_20b_math_16bit", tokenizer, save_method="merged_16bit")

# Push to Hugging Face Hub (uncomment and add your token)
# model.push_to_hub_merged("your-username/gpt-oss-20b-math", tokenizer, token="hf_...", save_method="lora")